# Klassifikationsbäume und Ensembles — Konzepte

Lies die Idee, die Formel und das kleine Beispiel in dieser Reihenfolge. Der Demonstrationscode wird bereitgestellt.

## Libraries and settings

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.metrics import confusion_matrix, roc_auc_score

## 1. Lies einen Klassifikationsbaum

Ein innerer Knoten stellt eine Ja-/Nein-Frage. Die **Wurzel** ist der erste Knoten, und ein **Blatt** ist ein Endknoten. Die Tiefe ist der längste Pfad von der Wurzel zu einem Blatt, gezählt in Kanten.

Ein Blatt schätzt Klassenwahrscheinlichkeiten anhand der Klassenanteile in den Trainingsdaten:
$$\hat P(y=k\mid\mathrm{Blatt})=\frac{\mathrm{Anzahl\ in\ Klasse}\ k}{\mathrm{Anzahl\ im\ Blatt}}.$$
Bei 145 Überlebenden und 125 Nichtüberlebenden gilt $P(\mathrm{überlebt})=145/270\approx0{,}537$, daher lautet die vorhergesagte Klasse „überlebt“. Ein Baum sagt sowohl Wahrscheinlichkeiten als auch Labels voraus.

## Gini: Wie gemischt ist ein Knoten?

Für die Klassenanteile $p_k$:
$$G=1-\sum_k p_k^2.$$
Für zwei Klassen gilt $G=1-p^2-(1-p)^2$. Ein reiner Knoten hat einen Gini-Wert von 0; eine gleichmäßige 50/50-Mischung hat einen Gini-Wert von 0,5. Die Entropie ist eine Alternative:
$$H=-\sum_k p_k\log(p_k),$$
wobei ein Anteil von null den Wert null beiträgt. Beide bevorzugen reinere Kindknoten.

Ein Split wird anhand der nach Größe gewichteten Unreinheit seiner Kindknoten bewertet:
$$\Delta G=G_{\mathrm{Elternknoten}}-\left(\frac{n_L}{n}G_L+\frac{n_R}{n}G_R\right).$$
$n_L,n_R$ sind die Größen der Kindknoten; $n=n_L+n_R$. Wähle die größte Verringerung und wiederhole dies anschließend rekursiv. Durchgerechnetes Beispiel: Der Elternknoten hat 40 Überlebende und 60 Nichtüberlebende ($G=0{,}48$). Der Split „männlich?“ ergibt einen gewichteten Gini-Wert von 0,36 und eine Verringerung von 0,12; der alternative Split nach erster Klasse ergibt eine kleinere Verringerung.

In [2]:
p = 0.4
print('Parent Gini:', 1 - p ** 2 - (1 - p) ** 2)
g_left = 1 - 0.2 ** 2 - 0.8 ** 2
g_right = 1 - 0.7 ** 2 - 0.3 ** 2
weighted = 0.6 * g_left + 0.4 * g_right
print('Weighted child Gini:', weighted)
print('Decrease:', 0.48 - weighted)

Parent Gini: 0.48
Weighted child Gini: 0.35999999999999993
Decrease: 0.12000000000000005


## 3. Wachsen lassen, stoppen und beschneiden

Probiere mögliche Eingaben und Schwellenwerte aus, behalte die größte Verringerung der Unreinheit und wiederhole dies in den Kindknoten. Stoppe, wenn ein Knoten rein ist oder eine Einstellung wie `max_depth` oder `min_samples_leaf` einen weiteren Split verhindert.

Pruning fügt wie bei Regressionsbäumen einen Strafterm pro Blatt hinzu:
$$\mathrm{Zielfunktion}=\mathrm{Baumunreinheit}+\alpha\times\mathrm{Anzahl\ der\ Blätter}.$$
Ein größeres `ccp_alpha` kann Äste entfernen. Tiefe Bäume können overfitten; wähle die Baumeinstellungen mithilfe von CV und nicht allein anhand der Trainings-Accuracy.

## 4. Random Forest und Boosting

Ein Random Forest erstellt viele Bäume aus Bootstrap-Stichproben und zufälligen Teilmengen von Merkmalen bei jedem Split. Scikit-learn bildet den Mittelwert der Klassenwahrscheinlichkeiten:
$$\hat p=\frac1B\sum_{b=1}^B\hat p_b.$$
Für 0,8, 0,6, 0,3 beträgt der Mittelwert ungefähr 0,567: Bei einem Schwellenwert von 0,5 wird „überlebt“ vorhergesagt. Klassisches Bagging wird häufig als Abstimmung beschrieben; die Mittelwertbildung von Klassenwahrscheinlichkeiten kann zu einem anderen Ergebnis führen als die Abstimmung anhand der Labels der Bäume.

Gradient Boosting erstellt kleine Bäume nacheinander, um die aktuellen Vorhersagen zu verbessern. Bei der Klassifikation arbeitet es mit Log-Loss und Pseudo-Residuen, nicht einfach mit den 0/1-Fehlern. Die Prinzipien sind dieselben wie bei der Regression, aber die Ausgaben beschreiben Klassenwahrscheinlichkeiten.

In [3]:
tree_probabilities = np.array([0.8, 0.6, 0.3])
print('Forest probability:', tree_probabilities.mean())
print('Predicted class:', int(tree_probabilities.mean() >= 0.5))

Forest probability: 0.5666666666666667
Predicted class: 1


## 5. Alles zusammenführen: eine kleine GridSearchCV

Das abschließende Beispiel kombiniert Trainings-/Testaufteilung, CV, einen Random Forest und AUC. `GridSearchCV`  probiert eine kleine Liste von Einstellungen aus und behält diejenige mit dem höchsten durchschnittlichen Validierungsscore.

In der Übung wird die gesamte Suche bereitgestellt: zwei `max_features`-Werte und drei `min_samples_leaf`-Werte, die mithilfe einer 5-fachen CV auf den Trainingszeilen evaluiert werden. Du erklärst das Ergebnis, anstatt die Suche selbst zu schreiben. Erst nach der Suche evaluieren wir auf den Testzeilen. AUC verwendet Wahrscheinlichkeiten; eine Konfusionsmatrix verwendet Labels bei einem Schwellenwert.

### Jupyter notebook — footer info

Include this information when submitting your notebook.

In [4]:
import platform
from datetime import datetime
print(platform.platform())
print("Python:", platform.python_version())
print("Date:", datetime.now().isoformat(timespec="seconds"))

Linux-6.8.0-1064-azure-x86_64-with-glibc2.41
Python: 3.11.17
Date: 2026-10-20T06:12:25
